# Python Insights - Analisando Dados com Python

## Case: Cancelamento de Clientes (Churn Analysis)

Você foi contratado por uma empresa com mais de 800 mil clientes para um projeto estratégico de Dados. Recentemente, a organização identificou que uma parcela significativa da base tornou-se inativa, indicando um alto índice de churn (cancelamentos).

**Objetivo:** Compreender os principais vetores de cancelamento e simular o impacto de ações corretivas para reduzir essa taxa e proteger a receita recorrente.

> **Nota Técnica:** Este notebook utiliza a arquitetura modular do repositório BI-Python. Os dados brutos estão localizados em `../data/raw/`.

---

### Roteiro de Análise
1.  **Diagnóstico Inicial:** Carregamento, tipagem e limpeza de dados.
2.  **Exploração Interativa:** Uso do *BI-Python Command Center* para visualização dinâmica de KPIs.
3.  **Análise de Causa Raiz:** Identificação de correlações entre variáveis operacionais e churn.
4.  **Simulação de Cenários:** Projeção de redução de churn baseada em ações de retenção.

In [1]:
import sys
!{sys.executable} -m pip install matplotlib pandas numpy ipywidgets


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\dell\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configuração de estilo global para consistência visual
plt.style.use('dark_background')
pd.set_option('display.max_columns', None)

# --- CARREGAMENTO INTELIGENTE DE DADOS ---
# Caminho relativo ajustado para a nova estrutura de pastas (notebooks -> data/raw)
CSV_PATH = '../data/raw/cancelamentos_sample.csv'

try:
    tabela = pd.read_csv(CSV_PATH)
    print(f"✅ Dataset carregado com sucesso: {tabela.shape[0]} linhas e {tabela.shape[1]} colunas.")
except FileNotFoundError:
    print(f"❌ ERRO CRÍTICO: Arquivo não encontrado em '{CSV_PATH}'.")
    print("Verifique se o arquivo 'cancelamentos_sample.csv' está na pasta 'data/raw/'.")
    tabela = pd.DataFrame() # Previne erros em cascata

# --- PRÉ-PROCESSAMENTO BÁSICO ---
if not tabela.empty:
    # Remoção de identificadores únicos que não agregam valor analítico
    if 'CustomerID' in tabela.columns:
        tabela.drop(columns=['CustomerID'], inplace=True)
    
    # Normalização da coluna alvo para garantir consistência nos filtros
    tabela['cancelou'] = tabela['cancelou'].astype(str).str.lower().map({
        'sim': True, 'não': False, 'nao': False, 
        'true': True, 'false': False, '1': True, '0': False
    })

    print("\n📊 Primeiras 5 observações:")
    display(tabela.head())
    
    print("\n🔍 Informações do Dataset:")
    tabela.info()

✅ Dataset carregado com sucesso: 50000 linhas e 12 colunas.

📊 Primeiras 5 observações:


,idade,sexo,tempo_como_cliente,frequencia_uso,ligacoes_callcenter,dias_atraso,assinatura,duracao_contrato,total_gasto,meses_ultima_interacao,cancelou
0,23.0,Male,13.0,22.0,2.0,1.0,Standard,Annual,909.58,23.0,NaN
1,49.0,Male,55.0,16.0,3.0,6.0,Premium,Monthly,207.00,29.0,NaN
2,30.0,Male,7.0,1.0,0.0,8.0,Basic,Annual,768.78,7.0,NaN
3,26.0,Male,40.0,5.0,3.0,8.0,Premium,Annual,398.00,12.0,NaN
4,27.0,Female,17.0,30.0,5.0,6.0,Basic,Annual,507.00,15.0,NaN



🔍 Informações do Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   idade                   50000 non-null  float64
 1   sexo                    49997 non-null  object 
 2   tempo_como_cliente      49998 non-null  float64
 3   frequencia_uso          50000 non-null  float64
 4   ligacoes_callcenter     50000 non-null  float64
 5   dias_atraso             50000 non-null  float64
 6   assinatura              50000 non-null  object 
 7   duracao_contrato        50000 non-null  object 
 8   total_gasto             50000 non-null  float64
 9   meses_ultima_interacao  50000 non-null  float64
 10  cancelou                0 non-null      object 
dtypes: float64(7), object(4)
memory usage: 4.2+ MB


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# --- CONFIGURAÇÃO DO TEMA DARK TECH ---
C = {
    'bg': '#0d1117', 'card': '#161b22', 
    'accent': '#00A8FF', 'text': '#c9d1d9', 
    'border': '#30363d', 'danger': '#ff5555'
}

# --- WIDGETS DE CONTROLE ---
btn_executar = widgets.Button(
    description='🔥 ATUALIZAR PAINEL', button_style='success', icon='bolt',
    layout=widgets.Layout(width='100%', height='45px', font_size='14px', font_weight='bold')
)

dropdown_kpi = widgets.Dropdown(
    options=[
        ('💰 Impacto Financeiro (Total Gasto)', 'total_gasto'),
        ('⏱️ Engajamento (Frequência de Uso)', 'frequencia_uso'),
        ('⚠️ Risco Operacional (Dias em Atraso)', 'dias_atraso'),
        ('📞 Volume de Suporte (Ligações Call Center)', 'ligacoes_callcenter'),
        ('📅 Tempo como Cliente', 'tempo_como_cliente')
    ],
    value='total_gasto',
    description='KPI Alvo:', style={'description_width': '100px'},
    layout=widgets.Layout(width='60%')
)

filtro_status = widgets.ToggleButtons(
    options=['Todos', 'Cancelados', 'Ativos'],
    value='Todos',
    description='Segmento:',
    button_style='', 
    tooltips=['Visão Geral', 'Apenas Churn', 'Apenas Retidos']
)

output_area = widgets.Output(layout=widgets.Layout(
    border=f'1px solid {C["border"]}', padding='20px', 
    background_color=C['card'], border_radius='12px', margin='20px 0'
))

# --- LÓGICA DE RENDERIZAÇÃO DINÂMICA (CORRIGIDA) ---
def gerar_analise(b):
    if tabela.empty: 
        with output_area: clear_output(); print("⚠️ Dataset vazio."); return
        
    with output_area:
        clear_output(wait=True)
        
        # Filtragem Inteligente
        if filtro_status.value == 'Cancelados': df_view = tabela[tabela['cancelou'] == True]
        elif filtro_status.value == 'Ativos': df_view = tabela[tabela['cancelou'] == False]
        else: df_view = tabela.copy()

        kpi = dropdown_kpi.value
        titulo_kpi = dropdown_kpi.label.split('(')[0].strip()
        
        # Header Visual LIMPO (Usando HTML direto, sem Markdown)
        header_html = f"""
        <div style="margin-bottom: 15px;">
            <h3 style="color:{C['accent']}; margin:0 0 5px 0; font-family:sans-serif;">
                📊 Análise Dinâmica: {titulo_kpi}
            </h3>
            <p style="color:{C['text']}; font-size:13px; margin:0; font-family:sans-serif;">
                Base: <strong>{len(df_view)} registros</strong> | Segmento: {filtro_status.value}
            </p>
        </div>
        """
        display(widgets.HTML(header_html))

        # Plotagem Estatística
        fig, ax = plt.subplots(figsize=(11, 5))
        fig.patch.set_facecolor(C['card']); ax.set_facecolor(C['card'])
        
        if kpi in df_view.columns:
            color_bar = C['danger'] if filtro_status.value == 'Cancelados' else C['accent']
            ax.hist(df_view[kpi].dropna(), bins=30, color=color_bar, alpha=0.85, edgecolor=C['bg'])
            ax.set_title(f'Distribuição de {titulo_kpi}', color=C['text'], fontsize=15, pad=15, fontfamily='sans-serif')
            
            # Linha de Média para Contexto
            media = df_view[kpi].mean()
            ax.axvline(media, color='#ffffff', linestyle='--', linewidth=1.5, label=f'Média: {media:.1f}')
            ax.legend(facecolor=C['card'], edgecolor=C['border'], labelcolor=C['text'])

        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_color(C['border'])
        ax.tick_params(colors=C['text'])
        plt.tight_layout()
        plt.show()

# --- MONTAGEM E EXECUÇÃO ---
btn_executar.on_click(gerar_analise)

painel = widgets.VBox([
    widgets.HTML(f"<h2 style='color:{C['accent']}; text-align:center; font-family:sans-serif;'>🎛️ BI-Python Command Center</h2>"),
    widgets.HBox([dropdown_kpi, filtro_status], layout=widgets.Layout(justify_content='center', gap='20px')),
    widgets.HTML("<div style='height:10px;'></div>"),
    btn_executar,
    output_area
])

display(painel)
gerar_analise(None)

In [4]:
# --- SIMULAÇÃO DE IMPACTO DE RETENÇÃO ---
# Criamos uma cópia para não corromper a base original de análise
tabela_simulada = tabela.copy()

print("🧪 Iniciando Simulação de Cenários de Retenção...\n")
print(f"Base Original: {len(tabela_simulada)} clientes | Taxa de Churn: {tabela_simulada['cancelou'].mean():.2%}")

# Ação 1: Migrar contratos mensais para anuais (removendo os mais propensos a sair)
antes = len(tabela_simulada)
tabela_simulada = tabela_simulada[tabela_simulada["duracao_contrato"] != "Monthly"]
print(f"➖ Após migração de contratos: {len(tabela_simulada)} clientes (-{antes - len(tabela_simulada)})")

# Ação 2: Resolução proativa de chamados (removendo clientes com >4 ligações)
antes = len(tabela_simulada)
tabela_simulada = tabela_simulada[tabela_simulada["ligacoes_callcenter"] <= 4]
print(f"➖ Após resolução de chamados: {len(tabela_simulada)} clientes (-{antes - len(tabela_simulada)})")

# Ação 3: Ação financeira para atrasos (removendo >20 dias de atraso)
antes = len(tabela_simulada)
tabela_simulada = tabela_simulada[tabela_simulada["dias_atraso"] <= 20]
print(f"➖ Após ação financeira: {len(tabela_simulada)} clientes (-{antes - len(tabela_simulada)})")

# Resultado Final da Simulação
taxa_final = tabela_simulada['cancelou'].mean()
reducao = ((tabela['cancelou'].mean() - taxa_final) / tabela['cancelou'].mean()) * 100

print(f"\n✅ RESULTADO DA SIMULAÇÃO:")
print(f"Nova Taxa de Churn: {taxa_final:.2%}")
print(f"Redução Relativa de Churn: {reducao:.1f}%")

🧪 Iniciando Simulação de Cenários de Retenção...

Base Original: 50000 clientes | Taxa de Churn: nan%
➖ Após migração de contratos: 40116 clientes (-9884)
➖ Após resolução de chamados: 29127 clientes (-10989)
➖ Após ação financeira: 26271 clientes (-2856)

✅ RESULTADO DA SIMULAÇÃO:
Nova Taxa de Churn: nan%
Redução Relativa de Churn: nan%
